# 03 — Training

Trains one backbone (VGG16 / MobileNet / DenseNet) on the MHI images
produced by `02_generate_MHI.ipynb`, using the two-stage transfer-learning
recipe in `src/training/trainer.py`:

1. **Stage 1 (warmup):** backbone frozen, only the new FC head trains.
2. **Stage 2 (fine-tuning):** top layers of the backbone unfrozen, whole
   network trains at a much lower learning rate.

Change `cfg.backbone` in the config cell below and re-run to train a
different architecture; run this notebook three times (once per backbone)
if you want to compare all three, as the base paper does.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt

from src.preprocessing.video_loader import build_dataset
from src.training.trainer import Trainer
from src.evaluation.plots import plot_training_curves
from src.utils.config import DEFAULT_CONFIG, MODELS_DIR, ensure_dirs
from src.utils.seed import set_global_seed

%matplotlib inline

## Configuration — edit these to try a different backbone or hyperparameters

In [ ]:
cfg = DEFAULT_CONFIG
cfg.backbone = "mobilenet"          # one of: "vgg16", "mobilenet", "densenet"
cfg.batch_size = 16
cfg.epochs_stage1 = 30
cfg.epochs_stage2 = 50
cfg.fine_tune = True
cfg.fine_tune_at_layer = -30        # unfreeze the last 30 backbone layers in stage 2

set_global_seed(cfg.random_seed)
ensure_dirs(cfg)
print(cfg)

## Load the preprocessed train / validation datasets

In [ ]:
train_ds, n_train = build_dataset("train", cfg)
val_ds, n_val = build_dataset("validation", cfg)
print(f"{n_train} training samples, {n_val} validation samples.")

If this cell raises `FileNotFoundError`, go back and run
`02_generate_MHI.ipynb` first — this notebook trains on
`data/processed/{train,validation}/`, which that notebook creates.

## Peek at a batch to confirm images/labels look right before training

In [ ]:
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i >= images.shape[0]:
        ax.axis("off")
        continue
    ax.imshow(images[i].numpy())
    label_idx = labels[i].numpy().argmax()
    ax.set_title(cfg.class_names[label_idx])
    ax.axis("off")
plt.tight_layout()
plt.show()

## Build the model and train (Stage 1 + Stage 2)

In [ ]:
trainer = Trainer(cfg)
trainer.model.summary()

In [ ]:
result = trainer.fit(train_ds, val_ds)

## Save the final model

In [ ]:
final_model_path = os.path.join(MODELS_DIR, f"{cfg.backbone}_final_model.keras")
result.model.save(final_model_path)
print("Saved:", final_model_path)

## Training curves (accuracy / loss, both stages stitched together)

In [ ]:
curve_path = plot_training_curves(result.history_stage1, result.history_stage2, cfg.backbone)
print("Saved plot to:", curve_path)

from IPython.display import Image, display
display(Image(filename=curve_path))

## Notes on reading the curves

- The dashed vertical line marks where fine-tuning (Stage 2) begins.
- If validation loss starts climbing right after that line while training
  loss keeps falling, the fine-tuning learning rate is probably too high,
  or `fine_tune_at_layer` is unfreezing too much of the backbone for how
  little data this dataset has — try a smaller `abs(fine_tune_at_layer)`
  or a lower `learning_rate_stage2` in `src/utils/config.py`.
- Early stopping (`EarlyStopping` in `src/training/callbacks.py`) already
  restores the best-validation-loss weights automatically, so the saved
  model is not necessarily the literal last epoch shown in the plot.

## Next step

Move on to **`04_evaluation.ipynb`** to evaluate this saved model on the
held-out test split.